# TP 5 — Vision par ordinateur et transfer learning

**M1 · Deep Learning & Big Data · Séance 5 · Environnement : Google Colab (GPU obligatoire)**

On veut reconnaître **5 espèces de fleurs** (marguerite, pissenlit, rose, tournesol, tulipe) à partir de photos, avec seulement quelques centaines d'images par espèce.

| Partie | Contenu |
|---|---|
| 1 | Charger et explorer les images |
| 2 | Un petit CNN entraîné de zéro |
| 3 | Le transfer learning : réutiliser un modèle entraîné sur ImageNet |
| 4 | Évaluer avec une matrice de confusion et analyser les erreurs |
| 5 | Bonus : fine-tuning et vos propres photos |

**Avant tout :** *Exécution › Modifier le type d'exécution › GPU*. Complétez les `____` des cellules `# TODO` et répondez aux questions ❓.

In [ ]:
import os, pathlib
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

print("GPU :", tf.config.list_physical_devices("GPU") or "⚠️ aucun : activez le GPU")

---
## Partie 1 — Charger et explorer les images

Les photos sont rangées dans **un dossier par espèce**. Keras sait transformer cette arborescence en jeu de données étiqueté : le nom du dossier devient l'étiquette.

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive = keras.utils.get_file(origin=URL, extract=True)

# On cherche le dossier qui contient les sous-dossiers d'espèces (son emplacement varie selon la version de Keras)
base = pathlib.Path(archive).parent
dossier = next(p.parent for p in base.rglob("daisy") if p.is_dir())
for espece in sorted(d.name for d in dossier.iterdir() if d.is_dir()):
    print(f"{espece:<12} {len(list((dossier / espece).glob('*.jpg')))} images")

In [ ]:
TAILLE = (160, 160)
LOT = 32

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    dossier,
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=TAILLE,
    batch_size=LOT,
)
classes = train_ds.class_names
print("Classes :", classes)

In [ ]:
plt.figure(figsize=(10, 6))
for images, etiquettes in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(classes[etiquettes[i]])
        plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Une image = un tableau de nombres
images, _ = next(iter(train_ds))
print("Forme d'un lot :", images.shape, "→ (images, hauteur, largeur, canaux R-V-B)")
print("Pixel en haut à gauche de la 1re image (R, V, B) :", images[0, 0, 0].numpy())

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

### ❓ Question 1.1

Combien de nombres faut-il pour décrire **une** image de 160 × 160 pixels en couleur ? Qu'est-ce que cela implique pour un réseau dense classique comme au TP 1 ?

*Votre réponse :*

---
## Partie 2 — Un petit CNN entraîné de zéro

Un **CNN** (réseau de neurones convolutif) enchaîne :
- des couches `Conv2D` : des **filtres** qui glissent sur l'image et repèrent des motifs (bords, textures…) ;
- des couches `MaxPooling2D` : elles **réduisent** l'image en gardant l'essentiel ;
- à la fin, des couches denses qui décident de la classe.

### 🛠️ Exercice 2.1

Complétez le modèle : la dernière couche doit avoir **un neurone par classe** et l'activation `softmax`.

In [ ]:
petit_cnn = keras.Sequential([
    keras.Input(shape=TAILLE + (3,)),
    layers.Rescaling(1.0 / 255),                 # pixels ramenés entre 0 et 1
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(____, activation=____),         # TODO
])
petit_cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
petit_cnn.summary()

In [ ]:
hist_cnn = petit_cnn.fit(train_ds, validation_data=val_ds, epochs=8)

In [ ]:
def tracer(hist, titre):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist.history["accuracy"], marker="o", label="Entraînement")
    ax[0].plot(hist.history["val_accuracy"], marker="o", label="Validation")
    ax[0].set_title("Bonnes réponses"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(hist.history["loss"], marker="o", label="Entraînement")
    ax[1].plot(hist.history["val_loss"], marker="o", label="Validation")
    ax[1].set_title("Perte"); ax[1].legend(); ax[1].grid(alpha=0.3)
    fig.suptitle(titre); plt.show()

tracer(hist_cnn, "Petit CNN entraîné de zéro")

### ❓ Question 2.2

Comparez les courbes d'entraînement et de validation. Que diagnostiquez-vous ? Pourquoi est-ce prévisible avec environ 3 000 images d'entraînement ?

*Votre réponse :*

---
## Partie 3 — Le transfer learning

**MobileNetV2** a été entraîné par Google sur ImageNet (plus d'un million d'images, 1 000 catégories). Ses couches de convolution savent déjà repérer bords, textures, formes et parties d'objets.

Plan :
1. on charge MobileNetV2 **sans sa dernière couche** (`include_top=False`) ;
2. on **gèle** ses poids (`trainable = False`) : on ne les modifie pas ;
3. on ajoute une petite **tête** de classification pour nos 5 fleurs, la seule partie entraînée.

On ajoute aussi de l'**augmentation de données** : à chaque époque, les images sont légèrement retournées, tournées, zoomées. Le modèle voit des variantes et généralise mieux.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

# Aperçu : 9 variantes d'une même image
images, _ = next(iter(train_ds))
plt.figure(figsize=(7, 7))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(augmentation(images[:1], training=True)[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Une image, neuf variantes"); plt.show()

### 🛠️ Exercice 3.1 — Charger et geler le modèle pré-entraîné

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=TAILLE + (3,),
    include_top=____,          # TODO : sans la couche de classification ImageNet
    weights="imagenet",
)
base.trainable = ____          # TODO : on gèle les poids

print("Couches dans MobileNetV2 :", len(base.layers))
print("Poids entraînables dans la base :", len(base.trainable_weights))

### 🛠️ Exercice 3.2 — Assembler le modèle complet

`preprocess_input` met les pixels à l'échelle attendue par MobileNetV2. `GlobalAveragePooling2D` résume chaque carte de caractéristiques en un seul nombre.

In [ ]:
entrees = keras.Input(shape=TAILLE + (3,))
x = augmentation(entrees)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
sorties = layers.Dense(____, activation="softmax")(x)     # TODO

transfert = keras.Model(entrees, sorties)
transfert.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Poids entraînables : {sum(np.prod(w.shape) for w in transfert.trainable_weights):,}".replace(",", " "))
print(f"Poids gelés        : {sum(np.prod(w.shape) for w in transfert.non_trainable_weights):,}".replace(",", " "))

In [ ]:
hist_tl = transfert.fit(train_ds, validation_data=val_ds, epochs=5)
tracer(hist_tl, "Transfer learning avec MobileNetV2")

In [ ]:
print(f"Petit CNN de zéro   : {petit_cnn.evaluate(val_ds, verbose=0)[1]:.1%}")
print(f"Transfer learning   : {transfert.evaluate(val_ds, verbose=0)[1]:.1%}")

### ❓ Question 3.3

Le modèle de transfert a des millions de poids gelés et seulement quelques milliers entraînés. Pourquoi est-il pourtant bien meilleur, et plus rapide à entraîner ?

*Votre réponse :*

---
## Partie 4 — Évaluer finement

Un pourcentage global cache beaucoup de choses. La **matrice de confusion** croise la vraie classe (lignes) et la classe prédite (colonnes) : la diagonale contient les bonnes réponses, le reste les confusions.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

vrais, predits, toutes_images = [], [], []
for images, etiquettes in val_ds:
    p = transfert.predict(images, verbose=0).argmax(axis=1)
    vrais.extend(etiquettes.numpy()); predits.extend(p); toutes_images.extend(images.numpy())
vrais, predits = np.array(vrais), np.array(predits)

ConfusionMatrixDisplay(confusion_matrix(vrais, predits), display_labels=classes).plot(cmap="Blues", xticks_rotation=45)
plt.title("Matrice de confusion (validation)"); plt.show()

print(classification_report(vrais, predits, target_names=classes))

### 🛠️ Exercice 4.1 — Regarder les erreurs

Affichez 8 images mal classées avec la vraie classe et la classe prédite.

In [ ]:
idx_erreurs = np.where(vrais != ____)[0]     # TODO : comparer aux prédictions
print(len(idx_erreurs), "erreurs sur", len(vrais))

plt.figure(figsize=(12, 6))
for k, i in enumerate(idx_erreurs[:8]):
    plt.subplot(2, 4, k + 1)
    plt.imshow(toutes_images[i].astype("uint8"))
    plt.title(f"vrai : {classes[vrais[i]]}\nprédit : {classes[predits[i]]}", color="red", fontsize=9)
    plt.axis("off")
plt.tight_layout(); plt.show()

### ❓ Question 4.2

Quelles espèces sont le plus souvent confondues ? En regardant les images mal classées, proposez deux explications.

*Votre réponse :*

---
## Partie 5 — Bonus

### 5.1 Fine-tuning

On dégèle les **dernières couches** de MobileNetV2 et on continue l'entraînement avec un **taux d'apprentissage 10 fois plus petit**, pour ajuster finement les connaissances sans les détruire.

In [ ]:
base.trainable = True
for couche in base.layers[:-30]:      # on ne dégèle que les 30 dernières couches
    couche.trainable = False

transfert.compile(optimizer=keras.optimizers.Adam(1e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_ft = transfert.fit(train_ds, validation_data=val_ds, epochs=3)
print(f"Après fine-tuning : {transfert.evaluate(val_ds, verbose=0)[1]:.1%}")

### 5.2 Vos propres photos

Importez une photo de fleur (ou de n'importe quoi d'autre, pour voir ce que fait le modèle face à une image hors sujet).

In [ ]:
from google.colab import files
for nom in files.upload():
    img = keras.utils.load_img(nom, target_size=TAILLE)
    probas = transfert.predict(np.expand_dims(keras.utils.img_to_array(img), 0), verbose=0)[0]
    plt.imshow(img); plt.axis("off"); plt.show()
    for i in probas.argsort()[::-1][:3]:
        print(f"  {classes[i]:<12} {probas[i]:.1%}")

### ❓ Question 5.3

Donnez une photo de chat au modèle. Que répond-il ? Qu'est-ce que cela révèle ?

*Votre réponse :*

---
## 🎯 Pour vérifier que vous avez compris

1. Quel est l'avantage d'une convolution par rapport à une couche dense pour une image ?
2. Qu'est-ce que « geler » un modèle pré-entraîné, et pourquoi le fait-on ?
3. Que montre une matrice de confusion qu'un pourcentage global ne montre pas ?

*Vos réponses :*